In [ ]:
# 
#
# PURPOSE:
# 1. Load the `phase1_prepared_data.pkl` file created by Notebook 01.
# 2. Standardize all data (fit scaler *only* on Healthy data).
# 3. Create a pool of "real faulty residuals" using the Moving Average trend.
#    (This was a key finding from our validation notebooks).
# 4. Generate a full set of synthetic faulty data using the "Bootstrap Crossover"
#    method, which our validation proved was the most realistic.
# 5. Create a 50/50 balanced dataset (Real Healthy vs. [Real Faulty + Synthetic Faulty]).
# 6. Train the Conv-BiLSTM classifier (with MC-Dropout) on this balanced data.
# 7. Save all critical artifacts (model, scaler, data) for the next notebook.

# --- Cell 1: Setup & Imports ---
import os, json, pickle, math, time, random, pathlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import welch, butter, filtfilt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve, roc_curve, confusion_matrix
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, UpSampling1D, Dense, Dropout, SpatialDropout1D,
                                     GlobalAveragePooling1D, Bidirectional, LSTM, Layer, TimeDistributed)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# --- Cell 2: Constants & File Paths ---
# All inputs are read from, and outputs are saved to, this single directory
OUT_DIR = Path("Final_Run_Outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Input file (from Notebook 01)
PKL_PATH = OUT_DIR / "phase1_prepared_data.pkl"

# Shapes from exploration
MAX_SEQ_LEN = 2048
N_CHANNELS = 23
SEED = 1337

# Synthesis parameters (from validation notebooks)
WIN = 256
HOP = 128

# --- Artifact Save/Load Points ---
# This .npz file will store all our processed data
ARTIFACTS_NPZ = OUT_DIR / "classification_pipeline_data.npz"
SCALER_FILE = OUT_DIR / "classifier_scaler.pkl"
MODEL_WEIGHTS_FILE = OUT_DIR / "classifier_mc_v2.weights.h5"
MODEL_INFERENCE_FILE = OUT_DIR / "classifier_mc_inference.keras"
SYNTH_DATA_FILE = OUT_DIR / "synthetic_positives_final.npy"

# --- Seeding ---
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
rng = np.random.default_rng(SEED)

print(f"Artifacts will be saved to: {OUT_DIR.resolve()}")

In [ ]:
# --- Cell 3: Load & Pad Raw Data ---
print(f"Loading data from {PKL_PATH}...")
with open(PKL_PATH, "rb") as f:
    P = pickle.load(f)

healthy = P["healthy_timeseries"]
faulty  = P["faulty_timeseries_for_gan"]
test_ts = P.get("test_timeseries", [])
test_y  = P.get("test_labels", None)

print(f"Loaded: {len(healthy)} healthy, {len(faulty)} faulty, {len(test_ts)} test series")

def pad_or_trim(x, T=MAX_SEQ_LEN):
    x = np.asarray(x, dtype=np.float32)
    # Ensure no NaNs or Infs from loading
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0) 
    Ti = len(x)
    if Ti >= T:
        return x[:T]
    out = np.zeros((T, x.shape[1]), dtype=np.float32)
    out[:Ti] = x
    if Ti > 0 and Ti < T:
        out[Ti:] = x[Ti-1] # Hold last value for padding
    return out

H = np.stack([pad_or_trim(x) for x in healthy], axis=0)
F = np.stack([pad_or_trim(x) for x in faulty],  axis=0)
T = np.stack([pad_or_trim(x) for x in test_ts], axis=0)

print(f"Padded Shapes -> H: {H.shape}  F: {F.shape}  T: {T.shape}")

In [ ]:
# --- Cell 4: Standardize Data (Fit on Healthy) ---
# This is a critical step. We fit the scaler *only* on the healthy data
# to prevent data leakage and establish a "normal" baseline.
print("Fitting StandardScaler on Healthy data...")
scaler = StandardScaler()
scaler.fit(H.reshape(-1, N_CHANNELS))

# Transform all datasets using the same scaler
H_scaled = scaler.transform(H.reshape(-1, N_CHANNELS)).reshape(H.shape)
F_scaled = scaler.transform(F.reshape(-1, N_CHANNELS)).reshape(F.shape)
T_scaled = scaler.transform(T.reshape(-1, N_CHANNELS)).reshape(T.shape)

# FIX: Convert string labels (e.g., 'before') to integers (0 or 1)
y_test_true = np.asarray([1 if label == 'before' else 0 for label in test_y]).astype(int).ravel()

print("Data standardized.")

# --- SAVE POINT ---
# We save all scaled data and the scaler. Notebook 03 will need these.
print(f"Saving scaled data and scaler...")
np.savez_compressed(
    ARTIFACTS_NPZ,
    H_scaled=H_scaled,
    F_scaled=F_scaled,
    T_scaled=T_scaled,
    y_test_true=y_test_true
)
with open(SCALER_FILE, "wb") as f:
    pickle.dump(scaler, f)
print(f"Saved data to {ARTIFACTS_NPZ}")
print(f"Saved scaler to {SCALER_FILE}")

In [ ]:
# --- Cell 5: Synthesis Helper Functions ---
# These functions are the final, validated versions from our experiments.
# We use Bootstrap Crossover (generate_residual_windows) and a clean
# composition pipeline (no soft_shrink, no texture_bed).

def moving_avg_trend(X, k=31):
    """Simple moving average trend extractor."""
    pad = k//2
    w = np.hanning(k).astype(np.float32)
    w = w / w.sum()
    out = np.zeros_like(X, dtype=np.float32)
    for n in range(X.shape[0]):
        for c in range(X.shape[2]):
            x = X[n,:,c]
            xpad = np.pad(x, (pad,pad), mode="edge")
            out[n,:,c] = np.convolve(xpad, w, mode="valid")
    return out

def generate_residual_windows(num, length=WIN, real_resid_pool=None):
    """
    Generates new, unique windows via Bootstrap Crossover.
    This method was found to be the most realistic in our validation.
    """
    N, T, C = real_resid_pool.shape
    out = []
    
    # 1. Create a flat pool of all possible windows
    window_pool = []
    for i in range(N):
        # Sample windows with a hop of 32 to get a dense pool
        for s in range(0, T - length + 1, 32): 
            window_pool.append(real_resid_pool[i, s:s+length])
    
    if not window_pool:
        raise ValueError("Residual pool is empty or too small for windowing.")
        
    window_pool = np.stack(window_pool)
    n_pool = len(window_pool)

    # 2. Generate 'num' new windows via crossover
    for _ in range(num):
        # Select two random parent windows
        idx_a, idx_b = rng.integers(0, n_pool, size=2)
        win_a = window_pool[idx_a]
        win_b = window_pool[idx_b]
        
        # Select a random crossover point
        crossover_point = rng.integers(int(length * 0.2), int(length * 0.8))
        
        # Create the new window
        new_win = np.concatenate(
            (win_a[:crossover_point, :], win_b[crossover_point:, :]), 
            axis=0
        )
        
        # Add a tiny bit of jitter to make it fully unique
        jitter = rng.normal(size=new_win.shape, scale=0.01).astype(np.float32)
        new_win += jitter
        
        out.append(new_win)
        
    return np.stack(out, axis=0)

def overlap_add(windows, full_len, hop):
    """Stitches overlapping windows back into a full sequence."""
    num, win_len, C = windows.shape
    win = np.hanning(win_len)[:, None]
    out = np.zeros((full_len, C), np.float32)
    wsm = np.zeros((full_len, 1), np.float32)
    t = 0
    for i in range(num):
        s, e = t, t+win_len
        if e > full_len: break
        out[s:e] += windows[i] * win
        wsm[s:e] += win
        t += hop
    wsm = np.maximum(wsm, 1e-6)
    return out / wsm

# --- Other helper functions for composition (PSD matching, etc.) ---
def _psd(x, fs=1.0):
    P, F = [], None
    for c in range(x.shape[1]):
        f, pxx = welch(x[:, c], fs=fs, nperseg=256, noverlap=128)
        F = f if F is None else F
        P.append(pxx)
    return F, np.mean(np.stack(P, 0), axis=0)

def hp_band_rms(x, fc=0.2, fs=1.0):
    b, a = butter(4, fc/(0.5*fs), btype="highpass")
    xf = np.stack([filtfilt(b, a, x[:,c], axis=0) for c in range(x.shape[1])], axis=1)
    return np.sqrt(np.mean(xf**2, axis=0))

def apply_calibration(gen_resid, real_resid, eps=1e-6, method="q95"):
    tgt = np.quantile(np.abs(real_resid), 0.95, axis=(0,1))
    gen = np.quantile(np.abs(gen_resid), 0.95, axis=0 if gen_resid.ndim==2 else (0,1))
    s = (tgt + eps) / (gen + eps)
    return gen_resid * s

def trend_gate(trend, sharp_k=8.0, z_k=1.2):
    g = np.abs(np.vstack([trend[0:1], np.diff(trend, axis=0)]))
    mu  = g.mean(axis=0, keepdims=True)
    sig = g.std(axis=0, keepdims=True) + 1e-6
    z = (g - (mu + z_k*sig)) / (sig); gate = 1.0 / (1.0 + np.exp(-sharp_k * z))
    k = 9; w = np.hanning(k)[:, None]; pad = k//2
    sm = np.concatenate([np.repeat(gate[:1], pad, 0), gate, np.repeat(gate[-1:], pad, 0)], 0)
    sm = np.stack([np.convolve(sm[:,c], w[:,0], mode='valid') for c in range(gate.shape[1])], 1) / w.sum()
    return np.clip(sm, 0, 1)

def premaintenance_ramp(T, start_frac=0.65, width=0.10):
    t = np.linspace(0,1,T); return (1/(1+np.exp(-(t-start_frac)/width)))[:,None]

def compose_synthetic_sequence(trend_base, real_resid_pool, full_len=MAX_SEQ_LEN, win=WIN, hop=HOP,
                               gate_blend=(0.75,0.25), gate_floor=0.10,
                               match_psd=True, band_lo=0.1, fs=1.0,
                               clip_q=None, global_scale=1.0):
    
    K = (full_len - win) // hop + 1
    resid_ws = generate_residual_windows(K, length=win, real_resid_pool=real_resid_pool)
    resid_full = overlap_add(resid_ws, full_len, hop=hop)

    tb = trend_base if trend_base.ndim==2 else trend_base[0]
    gt = trend_gate(tb, sharp_k=8.0, z_k=1.2)
    gr = premaintenance_ramp(full_len, start_frac=0.65, width=0.10)
    a,b = gate_blend; gate = np.clip(a*gt + b*gr, gate_floor, 1.0); resid_full *= gate

    # NOTE: soft_shrink and texture_bed are REMOVED as per our validation experiments.
    
    if match_psd:
        f, P_real = _psd(real_resid_pool.reshape(-1, real_resid_pool.shape[-1]), fs=fs)
        _, P_gen  = _psd(resid_full, fs=fs)
        band = f >= band_lo; r_power = np.median((P_gen[band]+1e-12)/(P_real[band]+1e-12))
        
        # This logic (from validation) gently boosts or trims power to match.
        if r_power > 1.3: s_g = 1.0/np.sqrt(r_power); s_g = np.clip(s_g, 0.85, 1.0); resid_full *= s_g
        elif r_power < 0.7: s_g = 1.0/np.sqrt(r_power); s_g = np.clip(s_g, 1.0, 1.15); resid_full *= s_g
            
        fc = max(band_lo, 0.30)
        real_rms = hp_band_rms(real_resid_pool.reshape(-1, real_resid_pool.shape[-1]), fc=fc, fs=fs)
        gen_rms  = hp_band_rms(resid_full, fc=fc, fs=fs)
        s_c = (real_rms+1e-6)/(gen_rms+1e-6); s_c = np.clip(s_c, 0.9, 1.1)
        resid_full *= s_c

    resid_full = apply_calibration(resid_full, real_resid_pool, method="q95")
    recon_scaled = tb[np.newaxis, ...] + resid_full[np.newaxis, ...]
    return recon_scaled, resid_full

print("All synthesis helper functions defined.")

In [ ]:
# --- Cell 6: Create Trend & Residual Pools ---
# This cell creates the two key components for synthesis:
# 1. H_tr: A pool of "normal" trends from healthy flights.
# 2. F_resid: A pool of "faulty" residuals from faulty flights.
print("Creating trend and residual pools using moving average...")

# --- LOAD POINT ---
try:
    data_archive = np.load(ARTIFACTS_NPZ)
    H_scaled = data_archive['H_scaled']
    F_scaled = data_archive['F_scaled']
    T_scaled = data_archive['T_scaled']
    y_test_true = data_archive['y_test_true']
    print("Loaded scaled data from NPZ file.")
except Exception as e:
    print(f"NPZ file not found ({e}). Using data from memory.")

# 1. Create Healthy Trend pool (base for synthesis)
H_tr = moving_avg_trend(H_scaled, k=31)

# 2. Create Real Faulty Residual pool (target for synthesis)
#    This is the key step: we use the "dumb" moving average to ensure
#    the subtle fault signature remains in the residual.
F_tr = moving_avg_trend(F_scaled, k=31)
F_resid = F_scaled - F_tr

print(f"Healthy Trend pool (H_tr) shape: {H_tr.shape}")
print(f"Real Faulty Residual pool (F_resid) shape: {F_resid.shape}")

# --- SAVE POINT ---
# Save these new artifacts to the NPZ file for use by other notebooks.
np.savez_compressed(
    ARTIFACTS_NPZ,
    H_scaled=H_scaled,
    F_scaled=F_scaled,
    T_scaled=T_scaled,
    y_test_true=y_test_true,
    H_tr=H_tr,
    F_resid=F_resid
)
print(f"Saved trends and residuals to {ARTIFACTS_NPZ}")

In [ ]:
# --- Cell 7: Generation Loop ---
# This is a long-running cell that generates all the synthetic data.
print("Starting synthetic data generation loop...")

# --- LOAD POINT ---
try:
    data_archive = np.load(ARTIFACTS_NPZ)
    H_tr = data_archive['H_tr']
    F_resid = data_archive['F_resid']
    F_scaled = data_archive['F_scaled'] 
    H_scaled = data_archive['H_scaled'] 
    print("Loaded trends and residuals from NPZ file.")
except Exception as e:
    print(f"NPZ file not found ({e}). Using data from memory.")

Nh, Nf = H_scaled.shape[0], F_scaled.shape[0]
need_syn = max(0, Nh - Nf)
print(f"Healthy={Nh}, Faulty(real)={Nf}, need synthetic positives={need_syn}")

# To run a quick test (e.g., 50 samples), uncomment the line below
# need_syn = 50 

Xs_list = []
remaining = need_syn
chunk_size = 256
rng = np.random.default_rng(SEED) # Re-init RNG for sampling

start_time = time.time()
while remaining > 0:
    n_batch = min(chunk_size, remaining)
    
    # Sample random HEALTHY trends to use as a base
    idx = rng.integers(0, len(H_tr), size=n_batch)
    trend_batch = H_tr[idx]
    
    batch_syn = []
    for i in range(n_batch):
        # Compose: Healthy Trend + Synthetic (Crossover) Faulty Residual
        x_syn, _ = compose_synthetic_sequence(
            trend_batch[i], 
            F_resid,
            gate_blend=(0.75, 0.25),
            gate_floor=0.10,
            match_psd=True, 
            band_lo=0.1
        )
        batch_syn.append(x_syn[0])
        
    Xs_list.append(np.stack(batch_syn, axis=0))
    remaining -= n_batch
    print(f"Generated {need_syn - remaining} / {need_syn} samples...")

if len(Xs_list) > 0:
    X_syn = np.concatenate(Xs_list, axis=0)
else:
    X_syn = np.empty((0, MAX_SEQ_LEN, N_CHANNELS), dtype=np.float32)

end_time = time.time()
print(f"\nSynthetic positives generated: {X_syn.shape}")
print(f"Generation took {end_time - start_time:.2f} seconds.")

# --- SAVE POINT ---
np.save(SYNTH_DATA_FILE, X_syn)
print(f"Saved synthetic data to {SYNTH_DATA_FILE}")

In [ ]:
# --- Cell 8: Create Balanced Dataset & Split ---
print("Creating balanced dataset for training...")

# --- LOAD POINT ---
try:
    data_archive = np.load(ARTIFACTS_NPZ)
    H_scaled = data_archive['H_scaled']
    F_scaled = data_archive['F_scaled']
    y_test_true = data_archive['y_test_true']
    T_scaled = data_archive['T_scaled']
    H_tr = data_archive['H_tr']
    F_resid = data_archive['F_resid']
    X_syn = np.load(SYNTH_DATA_FILE)
    print("Loaded all required data for balancing.")
except Exception as e:
    print(f"Data files not found ({e}). You may need to run previous steps.")

# X_pos = Real Faulty (30) + Synthetic Faulty (8706)
X_pos = np.concatenate([F_scaled, X_syn], axis=0)
y_pos = np.ones((X_pos.shape[0],), dtype=np.int32)

# X_neg = Healthy (8736)
X_neg = H_scaled
y_neg = np.zeros((X_neg.shape[0],), dtype=np.int32)

# Combine into one big dataset (17472 samples)
X_all = np.concatenate([X_pos, X_neg], axis=0)
y_all = np.concatenate([y_pos, y_neg], axis=0)

# Shuffle
indices = np.arange(len(X_all))
rng.shuffle(indices)
X_all = X_all[indices]
y_all = y_all[indices]

print(f"Balanced train pool: {X_all.shape}, {y_all.shape}")
print(f"Positive samples: {np.sum(y_all)}, Negative samples: {len(y_all) - np.sum(y_all)}")

# Split into Train (80%) and Validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, 
    test_size=0.20, 
    stratify=y_all, 
    random_state=SEED
)
print(f"Train shapes: {X_train.shape}, {y_train.shape}")
print(f"Val shapes:   {X_val.shape}, {y_val.shape}")
print(f"Test shapes:  {T_scaled.shape}, {y_test_true.shape}")

# --- SAVE POINT ---
# Save the final splits. This is the last save for this notebook.
print("Saving final train/val splits...")
np.savez_compressed(
    ARTIFACTS_NPZ,
    H_scaled=H_scaled, F_scaled=F_scaled, T_scaled=T_scaled, y_test_true=y_test_true,
    H_tr=H_tr, F_resid=F_resid,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val
)
print(f"Train/val data saved to {ARTIFACTS_NPZ}")

In [ ]:
# --- Cell 9: Define Classifier (with MC-Dropout) ---
# This custom layer is needed to force dropout at inference time for UQ
class MCDropout(Dropout):
    def call(self, inputs, training=None):
        return super().call(inputs, training=True) # Always True

def build_classifier(T=MAX_SEQ_LEN, C=N_CHANNELS):
    """
    Builds two versions of the model:
    1. model_train: For training, with standard dropout.
    2. model_inference: For UQ, with MC-Dropout layers.
    They will share the same weights.
    """
    inp = Input(shape=(T,C))
    
    # --- Define Shared Layers ---
    conv1 = Conv1D(32, 7, padding="same", activation="relu", name="conv1")
    pool1 = MaxPooling1D(4, name="pool1")
    conv2 = Conv1D(64, 5, padding="same", activation="relu", name="conv2")
    pool2 = MaxPooling1D(4, name="pool2")
    out_dense = Dense(1, activation="sigmoid", name="out")

    # --- Training Model ---
    # We use standard layers here
    x_train = conv1(inp)
    x_train = pool1(x_train)
    x_train = SpatialDropout1D(0.2, name="sp_drop1_train")(x_train)
    x_train = conv2(x_train)
    x_train = pool2(x_train)
    x_train = SpatialDropout1D(0.2, name="sp_drop2_train")(x_train)
    x_train = Bidirectional(LSTM(64, return_sequences=False, dropout=0.3, recurrent_dropout=0.3), name="bilstm_train")(x_train)
    x_train = Dropout(0.4, name="drop_final_train")(x_train)
    out_train = out_dense(x_train)
    
    model_train = Model(inp, out_train)
    model_train.compile(optimizer=Adam(learning_rate=1e-3), 
                        loss="binary_crossentropy", 
                        metrics=["accuracy", tf.keras.metrics.AUC(name="auroc")])
    
    # --- Inference Model (for UQ) ---
    # We re-build the graph using the *shared* layers but
    # swap Dropout for MCDropout to force it on at inference.
    x_inf = conv1(inp)
    x_inf = pool1(x_inf)
    x_inf = SpatialDropout1D(0.2, name="sp_drop1_inf")(x_inf, training=True) # Force on
    x_inf = conv2(x_inf)
    x_inf = pool2(x_inf)
    x_inf = SpatialDropout1D(0.2, name="sp_drop2_inf")(x_inf, training=True) # Force on
    x_inf = Bidirectional(LSTM(64, return_sequences=False, dropout=0.3, recurrent_dropout=0.3), name="bilstm_inf")(x_inf, training=True) # Force on
    x_inf = MCDropout(0.4, name="mc_drop_final_inf")(x_inf) # This layer is always on
    out_inf = out_dense(x_inf)
    
    model_inference = Model(inp, out_inf)
    
    return model_train, model_inference

clf_train, clf_inference = build_classifier()
print("\nClassifier for Training:")
clf_train.summary()

In [ ]:
# --- Cell 10: Train Classifier ---
# This is the second long-running cell.
print("Loading data for classifier training...")

# --- LOAD POINT ---
try:
    data_archive = np.load(ARTIFACTS_NPZ)
    X_train = data_archive['X_train']
    y_train = data_archive['y_train']
    X_val = data_archive['X_val']
    y_val = data_archive['y_val']
    print("Loaded Train/Val splits from NPZ.")
except Exception as e:
    print(f"NPZ file not found ({e}). Using data from memory.")

callbacks = [
    ReduceLROnPlateau(monitor="val_auroc", mode="max", factor=0.5, patience=4, verbose=1, min_lr=1e-5),
    EarlyStopping(monitor="val_auroc", mode="max", patience=10, restore_best_weights=True, verbose=1)
]

print("\n--- Starting Classifier Training ---")
hist = clf_train.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val), 
    epochs=50, 
    batch_size=32, 
    verbose=1, 
    callbacks=callbacks
)

# --- SAVE POINT ---
# After training, save the weights.
clf_train.save_weights(MODEL_WEIGHTS_FILE)
print(f"Saved classifier weights to {MODEL_WEIGHTS_FILE}")

In [ ]:
# Load the trained weights into the inference model and save the *entire* model.
# This is the final classifier model we will use in Notebook 03.
clf_inference.load_weights(MODEL_WEIGHTS_FILE)
clf_inference.save(MODEL_INFERENCE_FILE)
print(f"Loaded weights into MC-Dropout model and saved to {MODEL_INFERENCE_FILE}")
print("\n--- Classifier training complete. ---")

In [ ]:
# --- Cell 11: Evaluation & UQ Analysis (on Test Set) ---
# This cell runs the final evaluation from this notebook.
print("\nLoading artifacts for final evaluation...")

# --- LOAD POINT ---
try:
    data_archive = np.load(ARTIFACTS_NPZ)
    T_scaled = data_archive['T_scaled']
    y_test_true = data_archive['y_test_true'].astype(int).ravel()
    
    # We must load the model with the custom object definition
    clf_inference = tf.keras.models.load_model(
        MODEL_INFERENCE_FILE, 
        custom_objects={'MCDropout': MCDropout}
    )
    
    print(f"Loaded test data ({T_scaled.shape}) and inference model.")
except Exception as e:
    print(f"Error loading models/data: {e}. You may need to run previous steps.")


def best_f1_threshold(y_true, y_prob):
    """Finds the best F1 threshold from prediction probabilities."""
    p, r, th = precision_recall_curve(y_true, y_prob)
    f1 = np.nan_to_num(2 * p * r / (p + r + 1e-9))
    i = np.argmax(f1)
    return th[i], f1[i]

def mc_predict_proba(model, X, n_pass=50):
    """Runs N stochastic passes to get mean probability and uncertainty (std dev)."""
    print(f"Running {n_pass} MC-Dropout passes...")
    preds = [model(X, training=True).numpy().ravel() for _ in range(n_pass)]
    P = np.stack(preds, axis=0)
    return P.mean(0), P.std(0)

# --- Run Evaluation on Test Set ---
print(f"\n--- Evaluating on Test Set ({len(T_scaled)} samples) ---")
p_test_mean, p_test_std = mc_predict_proba(clf_inference, T_scaled)

# Calculate metrics
auroc_test = roc_auc_score(y_test_true, p_test_mean)
auprc_test = average_precision_score(y_test_true, p_test_mean)
best_thr, best_f1 = best_f1_threshold(y_test_true, p_test_mean)
y_test_pred = (p_test_mean >= best_thr).astype(int)

# FIX: Cast all metrics to standard Python floats for JSON
metrics = dict(
    auroc=float(auroc_test), 
    auprc=float(auprc_test), 
    best_f1=float(best_f1), 
    best_thr=float(best_thr)
)
print(f"\nTest Set Metrics (Balanced Training):")
print(json.dumps(metrics, indent=2))

In [ ]:
# --- SAVE POINT (Metrics & Predictions) ---
with open(OUT_DIR / "final_test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save the raw predictions to the NPZ for Notebook 03
print("Updating NPZ file with final predictions...")
np.savez_compressed(
    ARTIFACTS_NPZ,
    # Re-save everything from before
    H_scaled=H_scaled, F_scaled=F_scaled, T_scaled=T_scaled, y_test_true=y_test_true,
    H_tr=H_tr, F_resid=F_resid,
    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
    # Add new prediction arrays
    p_test_mean=p_test_mean,
    p_test_std=p_test_std,
    y_test_pred=y_test_pred
)
print(f"Saved final metrics and predictions to {OUT_DIR}")


# --- Cell 12: Plot Confusion Matrix & UQ ---
# These are the plots from your final successful run.
print("Generating final plots...")

# Plot 1: Confusion Matrix
cm = confusion_matrix(y_test_true, y_test_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Pred Healthy', 'Pred Faulty'], 
            yticklabels=['True Healthy', 'True Faulty'])
plt.title('Classifier Confusion Matrix (at